# CCA2 – CIFAR-10 CNN in **PyTorch**
**Before running:** menu *Runtime → Change runtime type → T4 GPU → Save*. Then *Runtime → Run all*.

At the end, a file `results_pytorch.json` downloads automatically, send it back to Claude.
**Screenshot** the output of the *Training* cell (Figure 1 of the report).

In [ ]:
# Step 1: Environment info
import time, sys, platform
t0 = time.time()
import torch, torchvision
import_time = time.time() - t0
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None (CPU)"
print("Python     :", sys.version.split()[0])
print("PyTorch    :", torch.__version__, "| torchvision", torchvision.__version__)
print("Import time: %.2f s" % import_time)
print("Device     :", gpu_name)
if gpu_name.startswith("None"): print("!!! NO GPU - go to Runtime > Change runtime type > T4 GPU, then Run all again !!!")

In [ ]:
# Step 2: Load CIFAR-10 and split 45,000 train / 5,000 val / 10,000 test
import numpy as np, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(42); np.random.seed(42)

t0 = time.time()
# Same CIFAR-10 file Keras uses (fast Google mirror; the default Toronto server can be very slow)
import urllib.request, tarfile, pickle, os
if not os.path.exists("cifar10.tar.gz"):
    urllib.request.urlretrieve("https://storage.googleapis.com/tensorflow/tf-keras-datasets/cifar-10-batches-py.tar.gz", "cifar10.tar.gz")
with tarfile.open("cifar10.tar.gz") as tar: tar.extractall("data")
def load_batch(path):
    with open(path, "rb") as f: d = pickle.load(f, encoding="bytes")
    x = torch.tensor(np.array(d[b"data"], dtype=np.uint8).reshape(-1, 3, 32, 32), dtype=torch.float32) / 255.0  # NCHW, 0-1
    return x, torch.tensor(d[b"labels"], dtype=torch.long)
parts = [load_batch(f"data/cifar-10-batches-py/data_batch_{i}") for i in range(1, 6)]
x_full = torch.cat([p[0] for p in parts]); y_full = torch.cat([p[1] for p in parts])
x_test, y_test = load_batch("data/cifar-10-batches-py/test_batch")
x_train, y_train = x_full[:45000], y_full[:45000]
x_val,   y_val   = x_full[45000:], y_full[45000:]
data_time = time.time() - t0

BATCH, EPOCHS, LR = 64, 10, 1e-3
train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(TensorDataset(x_val, y_val), batch_size=256)
test_loader  = DataLoader(TensorDataset(x_test, y_test), batch_size=256)
print("Train/Val/Test:", len(x_train), len(x_val), len(x_test), "| data load %.1f s" % data_time)

In [ ]:
# Step 3: Model, loss, optimizer   (KEY CODE for the report)
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128 * 4 * 4, 128), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(128, 10))
    def forward(self, x):
        return self.classifier(self.features(x))

model = CNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print("Total parameters:", n_params)

In [ ]:
# Step 4: Training   (SCREENSHOT THIS CELL'S OUTPUT -> Figure 1)
def evaluate(loader):
    model.eval(); correct = total = 0; loss_sum = 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += loss_fn(out, y).item() * len(y)
            correct += (out.argmax(1) == y).sum().item(); total += len(y)
    return loss_sum / total, correct / total

def sync():
    if device.type == "cuda": torch.cuda.synchronize()

history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": [], "epoch_time": []}
if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
start = time.time()
for epoch in range(EPOCHS):
    model.train(); t = time.time(); correct = total = 0; loss_sum = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * len(y)
        correct += (out.argmax(1) == y).sum().item(); total += len(y)
    val_loss, val_acc = evaluate(val_loader)
    sync(); ep_time = time.time() - t
    for k, v in zip(history, [loss_sum / total, correct / total, val_loss, val_acc, ep_time]):
        history[k].append(v)
    print(f"Epoch {epoch+1:2d}/{EPOCHS} - {ep_time:5.1f}s - loss: {loss_sum/total:.4f} - "
          f"accuracy: {correct/total:.4f} - val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}")
train_time = time.time() - start
print("Total training time: %.1f s" % train_time)

In [ ]:
# Step 5: Test accuracy, inference speed, memory
test_loss, test_acc = evaluate(test_loader)
model.eval()
with torch.no_grad():
    x_dev = x_test.to(device)
    for _ in range(3): model(x_dev[:256])          # warm-up
    sync(); t = time.time(); preds = []
    for i in range(0, len(x_dev), 256):
        preds.append(model(x_dev[i:i + 256]).argmax(1))
    sync(); infer_time = time.time() - t
preds = torch.cat(preds).cpu().numpy()
peak_mem = torch.cuda.max_memory_allocated() / 1e6 if device.type == "cuda" else None
print("Test loss %.4f | Test accuracy %.4f" % (test_loss, test_acc))
print("Inference: %.2f s for 10,000 images = %.4f ms/image" % (infer_time, infer_time / 10000 * 1000))
print("Peak GPU memory:", None if peak_mem is None else "%.0f MB" % peak_mem)

In [ ]:
# Step 5b: Fresh-install test for the 'Installation and setup' comparison (max 10 min, then skipped)
# Installs PyTorch + torchvision into a separate folder (does not touch the running environment)
import subprocess, shutil, os
install_time = install_size_mb = None
try:
    shutil.rmtree("/tmp/fresh_pkgs", ignore_errors=True)
    t0 = time.time()
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
                        "--target", "/tmp/fresh_pkgs", "torch", "torchvision"], capture_output=True, text=True, timeout=600)
    install_time = time.time() - t0
    install_size_mb = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk("/tmp/fresh_pkgs") for f in fs) / 1e6
    print("Fresh install time: %.0f s | size on disk: %.0f MB" % (install_time, install_size_mb))
    if r.returncode != 0: print(r.stderr[-1500:])
    shutil.rmtree("/tmp/fresh_pkgs", ignore_errors=True)
except Exception as e:
    print("Install test skipped:", e)

In [ ]:
# Step 6: Plots + save results (send results_pytorch.json to Claude)
import json, matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ep = range(1, EPOCHS + 1)
ax[0].plot(ep, history["accuracy"], label="train"); ax[0].plot(ep, history["val_accuracy"], label="val"); ax[0].set_title("PyTorch accuracy")
ax[1].plot(ep, history["loss"], label="train");     ax[1].plot(ep, history["val_loss"], label="val");     ax[1].set_title("PyTorch loss")
for a in ax: a.set_xlabel("epoch"); a.legend()
plt.show()

results = {"framework": "PyTorch", "framework_version": torch.__version__, "python": sys.version.split()[0],
           "device": gpu_name, "import_time_s": import_time, "install_time_s": install_time, "install_size_mb": install_size_mb, "data_load_s": data_time, "params": n_params,
           "batch": BATCH, "epochs": EPOCHS, "lr": LR, "history": history, "train_time_s": train_time,
           "test_loss": test_loss, "test_acc": test_acc, "inference_total_s": infer_time,
           "inference_ms_per_image": infer_time / 10000 * 1000, "peak_mem_mb": peak_mem,
           "preds": preds.tolist(), "labels": y_test.tolist()}
with open("results_pytorch.json", "w") as f: json.dump(results, f)
try:
    from google.colab import files; files.download("results_pytorch.json")
except Exception as e:
    print("Download manually from the Files panel:", e)